# Challenge 4: Programming an Agent Workflow
A sequential workflow with Search, Critique, and Refine agents that answers questions
by finding data, verifying it, and refining the response before returning it.

In [12]:
!pip install google-adk requests

## Setup imports and environment

In [13]:
import os
import logging
import time
from typing import Optional
from google.adk import Agent
from google.adk.agents import SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from google.adk.tools import google_search

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-01-fab96dd167ae"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.getLogger("google_genai.types").setLevel(logging.ERROR)

## Greeter Agent
Receives the user's question and passes it along to the workflow.

In [14]:
greeter = Agent(
    name="greeter",
    model="gemini-2.0-flash-lite",
    description="Greets the user and acknowledges their question.",
    instruction="""You are the first agent in a research workflow.
Acknowledge the user's question and restate it clearly so the next agent knows exactly what to research.
Do NOT answer the question yourself. Just clarify what needs to be researched.""",
)

## Search Agent
Finds data to answer the question using Google Search.

In [15]:
search_agent = Agent(
    name="search_agent",
    model="gemini-2.0-flash-lite",
    description="Researches and answers questions using Google Search.",
    instruction="""You are a research specialist. Use Google Search to find accurate, up-to-date information
to answer the question from the previous agent. Provide a detailed answer with key facts and sources.""",
    tools=[google_search],
)

## Critique Agent
Reviews the search agent's response and suggests improvements.

In [16]:
critique_agent = Agent(
    name="critique_agent",
    model="gemini-2.0-flash-lite",
    description="Reviews responses and suggests improvements.",
    instruction="""You are a critical reviewer. Look at the previous agent's response and evaluate it:
1. Is the information accurate and complete?
2. Is anything missing or unclear?
3. Could the response be better organized or more concise?

Provide specific suggestions for how the response can be improved.
Do NOT rewrite the response yourself. Just provide constructive feedback.""",
)

## Refine Agent
Rewrites the response based on the critique agent's feedback.

In [17]:
refine_agent = Agent(
    name="refine_agent",
    model="gemini-2.0-flash-lite",
    description="Refines and improves responses based on feedback.",
    instruction="""You are a writing specialist. Take the original response from the search agent
and the feedback from the critique agent, and produce a final, polished answer.
The final answer should be:
- Accurate and well-organized
- Clear and easy to understand
- Concise but complete

Write the final answer directly to the user.""",
)

## Sequential Workflow
Chains the agents: Greeter -> Search -> Critique -> Refine

In [18]:
workflow = SequentialAgent(
    name="research_workflow",
    description="A sequential workflow that researches, critiques, and refines answers.",
    sub_agents=[greeter, search_agent, critique_agent, refine_agent],
)

## Helper to run the workflow and display all events
Shows output from each sub-agent in the sequence.

In [19]:
async def run_workflow_verbose(agent, user_message: str) -> str:
    """Run the workflow, print events from each sub-agent, and return the final response."""
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name=agent.name, user_id="test_user"
    )
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)

    content = types.Content(
        role="user", parts=[types.Part(text=user_message)]
    )
    final_response = ""
    async for event in runner.run_async(
        user_id="test_user", session_id=session.id, new_message=content
    ):
        # Show which agent is producing output
        if event.author and event.content and event.content.parts:
            txt = event.content.parts[0].text
            if txt:
                print(f"\n--- [{event.author}] ---")
                print(txt.strip()[:500])  # Truncate long outputs for readability
        if event.is_final_response() and event.content and event.content.parts:
            final_response = event.content.parts[0].text
    return final_response

## Test: Run questions through the workflow
Each query goes through Greeter -> Search -> Critique -> Refine

In [20]:
test_queries = [
    "What are the benefits of renewable energy?",
    "Explain how large language models work.",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    result = await run_workflow_verbose(workflow, query)
    print(f"\n*** FINAL REFINED ANSWER ***")
    print(result)
    time.sleep(5)


Query: What are the benefits of renewable energy?

--- [greeter] ---
Okay, I understand. The question is: What are the benefits of renewable energy?

--- [search_agent] ---
The benefits of renewable energy are numerous and far-reaching, positively impacting the environment, economy, and public health. Here's a breakdown:

*   **Environmental Advantages**:
    *   **Reduced emissions**: Renewable sources like solar and wind produce little to no greenhouse gas emissions, unlike fossil fuels.
    *   **Cleaner air and water**: Renewable energy reduces air and water pollution, as it doesn't create waste or contamination risks.
    *   **Conservation of resources**: Ren

--- [critique_agent] ---
Here's a critical review of the response:

1.  **Accuracy and Completeness:** The information is accurate and covers the major benefits of renewable energy. The categories and the points within each are all correct. It is a good overview.

2.  **Missing or Unclear:** The response could be slightly 

## Loop Workflow: Iterative Q&A Refinement
A LoopAgent that repeatedly critiques and refines an answer until quality is acceptable,
using exit_loop to break out when done.

In [25]:
from google.adk.agents import LoopAgent
from google.adk.tools import exit_loop

drafter = Agent(
    name="drafter",
    model="gemini-2.0-flash-lite",
    description="Drafts and revises answers based on feedback.",
    instruction="""You are a writer. On the first pass, write an initial answer to the question.
On subsequent passes, revise your answer based on the critic's feedback.
Keep responses concise (under 200 words).""",
    tools=[google_search],
)

critic = Agent(
    name="critic",
    model="gemini-2.0-flash-lite",
    description="Reviews drafts and decides if they are good enough.",
    instruction="""You are a critic. Review the drafter's response.
If the answer is accurate, well-organized, and complete, call the exit_loop tool to stop.
If it needs improvement, provide specific feedback for the drafter to address in the next iteration.
Be concise in your feedback.""",
    tools=[exit_loop],
)

loop_refinement = LoopAgent(
    name="iterative_refinement",
    description="Iteratively drafts and refines answers until quality is met.",
    sub_agents=[drafter, critic],
    max_iterations=3,
)

# Final presenter takes the refined content and presents it
presenter = Agent(
    name="presenter",
    model="gemini-2.0-flash-lite",
    description="Presents the final refined answer.",
    instruction="""Take the draft from the previous agents and present it as a clean, final answer to the user.
Do not add new information, just present what was already written in a polished way.""",
)

loop_workflow = SequentialAgent(
    name="loop_workflow",
    description="Iteratively refines an answer, then presents it.",
    sub_agents=[loop_refinement, presenter],
)

## Test: Loop Workflow

In [26]:
print("\n" + "="*60)
print("LOOP WORKFLOW TEST")
print("="*60)
result = await run_workflow_verbose(loop_workflow, "What causes earthquakes and how are they measured?")
print(f"\n*** FINAL LOOP ANSWER ***")
print(result)


LOOP WORKFLOW TEST

--- [drafter] ---
Earthquakes are primarily caused by the sudden release of energy in the Earth's lithosphere, creating seismic waves. This energy release is usually due to the movement of tectonic plates, which constantly shift and interact. As these plates move, they can get stuck, building up stress over time. When the stress exceeds the strength of the rocks, they rupture along a fault line, causing an earthquake.

Earthquakes are measured using seismographs, which detect and record the ground motion. The dat

--- [critic] ---
The answer is accurate, well-organized, and complete.

--- [presenter] ---
Earthquakes are primarily caused by the sudden release of energy in the Earth's lithosphere, generating seismic waves. This energy release is typically due to the movement of tectonic plates, which are constantly shifting and interacting. As these plates move, they can get stuck, building up stress over time. When the stress exceeds the rocks' strength, they ruptur

## Parallel Workflow: Research Multiple Topics Simultaneously
A ParallelAgent that runs multiple research agents at the same time,
then a summary agent combines the results.

In [23]:
from google.adk.agents import ParallelAgent

# Two parallel researchers
benefits_researcher = Agent(
    name="benefits_researcher",
    model="gemini-2.0-flash-lite",
    description="Researches benefits and advantages.",
    instruction="""Research and list the key BENEFITS and ADVANTAGES of the topic.
Be concise, use bullet points. Max 150 words.""",
    tools=[google_search],
)

risks_researcher = Agent(
    name="risks_researcher",
    model="gemini-2.0-flash-lite",
    description="Researches risks and challenges.",
    instruction="""Research and list the key RISKS and CHALLENGES of the topic.
Be concise, use bullet points. Max 150 words.""",
    tools=[google_search],
)

parallel_research = ParallelAgent(
    name="parallel_research",
    description="Runs benefit and risk research in parallel.",
    sub_agents=[benefits_researcher, risks_researcher],
)

# Summary agent combines parallel results
summarizer = Agent(
    name="summarizer",
    model="gemini-2.0-flash-lite",
    description="Combines research from multiple agents into a balanced summary.",
    instruction="""You are a summarizer. Combine the benefits and risks research from the previous agents
into a single balanced summary. Present both sides clearly.
Format: Benefits section, then Risks section, then a brief conclusion.""",
)

# Full workflow: parallel research then summarize
parallel_workflow = SequentialAgent(
    name="balanced_research_workflow",
    description="Researches benefits and risks in parallel, then summarizes.",
    sub_agents=[parallel_research, summarizer],
)

## Test: Parallel Workflow

In [24]:
print("\n" + "="*60)
print("PARALLEL WORKFLOW TEST")
print("="*60)
result = await run_workflow_verbose(parallel_workflow, "What is the impact of artificial intelligence on the job market?")
print(f"\n*** FINAL PARALLEL ANSWER ***")
print(result)


PARALLEL WORKFLOW TEST

--- [benefits_researcher] ---
The impact of Artificial Intelligence (AI) on the job market is multifaceted:

*   **Job Creation:** AI is expected to create millions of new jobs in fields like data analytics, machine learning, and AI development.
*   **Job Displacement:** AI can automate routine tasks, potentially leading to job losses, particularly in industries relying on repetitive work.
*   **Job Transformation:** AI is enhancing existing jobs by improving accuracy and efficiency, such as in healthcare and data analysis.


--- [risks_researcher] ---
The impact of Artificial Intelligence (AI) on the job market is multifaceted, presenting both risks and challenges. Here's a concise overview:

*   **Job Displacement:** AI can automate repetitive or routine tasks, leading to job losses, particularly in customer service, data analysis, and entry-level positions. Some estimates suggest millions of jobs could be affected.
*   **Job Creation:** AI also drives the cr